In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import warnings 


warnings.filterwarnings("ignore")

In [ ]:
#loading the summary csv file
df = pd.read_csv("../results/dataset_summary.csv")

print(df.head())
print(df.columns.tolist())


In [ ]:
# Clean, minimal plot style
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})

In [ ]:
#dataset Overview Table 
overview = df[[
    'dataset_name', 'suite_name', 'task_type',
    'n_samples', 'n_features', 'n_categorical',
    'n_numerical', 'n_classes', 'has_missing'
]].copy()

overview = overview.sort_values(
    ['suite_name', 'n_samples'],
    ascending=[True, False]
).reset_index(drop=True)

print(overview.to_string())

In [ ]:
#suite and task type distribution

# The paper expects: 15 numerical_clf, 20 numerical_reg, 7 categorical_clf, 13 categorical_reg.

suite_counts = df['suite_name'].value_counts()
print(suite_counts)

# Simple bar chart to visualize the distribution
suite_counts.plot(kind='bar')
plt.title('Dataset Count per Suite')
plt.xlabel('Suite')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.savefig('../results/suite_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Sample size distribution

df['n_samples'].hist(bins=20)
plt.axvline(10000, color='red', linestyle='--', label='Cap at 10k')
plt.title('Sample Size Distribution')
plt.xlabel('Number of Samples')
plt.ylabel('Count')
plt.legend()
plt.savefig('../results/sample_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(df['n_samples'].describe().round(0))
print(f"\nDatasets above 10,000 samples (need capping): {(df['n_samples'] > 10000).sum()}")

In [ ]:
#Feature Count and Dimensionality Ratio

# Compute d/n ratio — measures how wide the dataset is relative to its size.
# The paper requires this to be below 0.10 for all datasets.
df['dim_ratio'] = df['n_features'] / df['n_samples']

# Look at the spread of feature counts across all datasets
df['n_features'].hist(bins=20, color='steelblue', edgecolor='white')
plt.title('Feature Count Distribution')
plt.xlabel('Number of Features')
plt.ylabel('Count')
plt.savefig('../results/feature_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Confirm the paper's dimensionality requirement holds for all datasets
print(df[['n_features', 'dim_ratio']].describe().round(4))
print(f"\nAll datasets below d/n < 0.10: {(df['dim_ratio'] < 0.10).all()}")
print(f"Max d/n ratio: {df['dim_ratio'].max():.4f}")

In [ ]:
# Categorical Feature Analysis

# Focus only on the categorical suites
cat_df = df[df['suite_name'].str.contains('categorical')].copy()

# What fraction of features are categorical in each dataset?
cat_df['pct_categorical'] = cat_df['n_categorical'] / cat_df['n_features'] * 100

# Look at the breakdown per dataset
print(cat_df[['dataset_name', 'n_features', 'n_categorical', 'n_numerical', 'pct_categorical']]
      .sort_values('pct_categorical', ascending=False)
      .to_string(index=False))

# Plot the distribution of categorical feature percentages
cat_df['pct_categorical'].hist(bins=10)
plt.title('% Categorical Features per Dataset (Categorical Suites)')
plt.xlabel('% of Features that are Categorical')
plt.ylabel('Count')
plt.savefig('../results/categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [18]:
# Group datasets by size and feature count.
# These bucket boundaries are chosen to roughly split the datasets into thirds.

def size_bucket(n):
    if n < 10000:  return 'small (<10k)'
    if n < 50000:  return 'medium (10k-50k)'
    return                'large (>50k)'

def feature_bucket(f):
    if f < 20:   return 'low (<20)'
    if f < 100:  return 'medium (20-100)'
    return               'high (>100)'

df['size_bucket']    = df['n_samples'].apply(size_bucket)
df['feature_bucket'] = df['n_features'].apply(feature_bucket)

# Check how datasets are distributed across buckets
print("Size buckets:")
print(df['size_bucket'].value_counts())

print("\nFeature buckets:")
print(df['feature_bucket'].value_counts())

# Save the enriched summary — this is what all future scripts will use
df.to_csv('../results/dataset_summary_enriched.csv', index=False)

Size buckets:
size_bucket
medium (10k-50k)    26
large (>50k)        15
small (<10k)        14
Name: count, dtype: int64

Feature buckets:
feature_bucket
low (<20)          36
medium (20-100)    17
high (>100)         2
Name: count, dtype: int64
